In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

### Create Spark Session

In [2]:
spark = SparkSession.builder.appName("basic").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 12:16:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Loads demo data

In [3]:
customers = spark.read.option("header", True).csv("../data/raw/customers.csv")

In [4]:
orders = spark.read.option("header", True).csv("../data/raw/orders.csv")

In [5]:
products = spark.read.option("header", True).csv("../data/raw/products.csv")

In [6]:
yellow_tripdata = spark.read.parquet("../data/raw/yellow_tripdata_2023-01.parquet")

In [ ]:
data = [
    (1, "Alice", "Bangladesh", 25),
    (2, "Bob", "United States", 34),
    (3, "Charlie", "Bangladesh", 42),
]

In [ ]:
df = spark.createDataFrame(data, ["customer_id", "name", "country", "age"])

In [ ]:
df.show()

In [ ]:
df.printSchema()

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.select("name", (F.col("age") + 1).alias("next_year_age")).show()

In [ ]:
df.filter((F.col("age") > 30) & (F.col("country") == "Bangladesh")).show()

In [ ]:
df.select("country").distinct().show()

In [ ]:
df.groupBy("country").agg(
    F.count("*").alias("customer_count"),
    F.sum("age").alias("total_age"),
    F.max("age").alias("max_age"),
    F.min("age").alias("min_age"),
).show()

In [ ]:
df.createOrReplaceTempView("customers")

In [ ]:
spark.sql("""
    SELECT
      country,
      COUNT(*) AS customer_count,
      SUM(age) AS total_age
    FROM customers
    GROUP BY country
""").show()

### Window Functions

In [ ]:
window = Window.partitionBy("country").orderBy(F.col("total_spent").desc())
result = customers.withColumn("rank", F.row_number().over(window)).filter(
    F.col("rank") <= 3
)
result.show()

### UDF

In [ ]:
def classify_age(age) -> str:
    if isinstance(age, str):
        age = int(age)
    if not age:
        return "unknown"
    if age < 18:
        return "minor"
    if age < 60:
        return "adult"
    return "senior"

In [ ]:
classify_age_udf = F.udf(classify_age, T.StringType())

In [ ]:
customers.withColumn("age_group", classify_age_udf(F.col("age"))).show()

In [8]:
result = customers.withColumn(
    "age_group",
    F.when(F.col("age") < 18, "minor")
    .when(F.col("age") < 60, "adult")
    .otherwise("senior"),
)

In [9]:
result.show()

+-----------+----------+----------+---+-----------+---------+
|customer_id|      name|   country|age|total_spent|age_group|
+-----------+----------+----------+---+-----------+---------+
|          1|      Yule|  Pakistan| 39|       4431|    adult|
|          2|   Emmalee|  Pakistan| 31|       6974|    adult|
|          3|   Angelle|     Nepal| 40|       4905|    adult|
|          4|   Osbourn|      Iran| 33|       4861|    adult|
|          5|     Trixy|     Nepal| 25|       5316|    adult|
|          6|     Orton|      Iran| 61|       1189|   senior|
|          7|     Mable|      Iran| 18|       8782|    adult|
|          8|    Zorine|      Iran| 58|       1622|    adult|
|          9|   Janette|     Nepal| 60|       4252|   senior|
|         10|  Sapphire|  Pakistan| 69|       9982|   senior|
|         11|    Allsun|      Iran| 51|        692|    adult|
|         12|    Fletch|  Pakistan| 39|       5287|    adult|
|         13|    Giorgi|      Iran| 48|       7205|    adult|
|       

### Inspecting Partitions

In [11]:
customers.rdd.getNumPartitions()

1

In [12]:
result.explain("formatted")

== Physical Plan ==
* Project (2)
+- Scan csv  (1)


(1) Scan csv 
Output [5]: [customer_id#17, name#18, country#19, age#20, total_spent#21]
Batched: false
Location: InMemoryFileIndex [file:/home/musleh/programming/languages/python/tutorials/pyspark-learning/data/raw/customers.csv]
ReadSchema: struct<customer_id:string,name:string,country:string,age:string,total_spent:string>

(2) Project [codegen id : 1]
Output [6]: [customer_id#17, name#18, country#19, age#20, total_spent#21, CASE WHEN (cast(age#20 as bigint) < 18) THEN minor WHEN (cast(age#20 as bigint) < 60) THEN adult ELSE senior END AS age_group#74]
Input [5]: [customer_id#17, name#18, country#19, age#20, total_spent#21]




In [7]:
yellow_tripdata.rdd.getNumPartitions()

12

In [8]:
yellow_tripdata.explain("formatted")

== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [19]: [VendorID#65L, tpep_pickup_datetime#66, tpep_dropoff_datetime#67, passenger_count#68, trip_distance#69, RatecodeID#70, store_and_fwd_flag#71, PULocationID#72L, DOLocationID#73L, payment_type#74L, fare_amount#75, extra#76, mta_tax#77, tip_amount#78, tolls_amount#79, improvement_surcharge#80, total_amount#81, congestion_surcharge#82, airport_fee#83]
Batched: true
Location: InMemoryFileIndex [file:/home/musleh/programming/languages/python/tutorials/pyspark-learning/data/raw/yellow_tripdata_2023-01.parquet]
ReadSchema: struct<VendorID:bigint,tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,passenger_count:double,trip_distance:double,RatecodeID:double,store_and_fwd_flag:string,PULocationID:bigint,DOLocationID:bigint,payment_type:bigint,fare_amount:double,extra:double,mta_tax:double,tip_amount:double,tolls_amount:double,improvement_surcharge:double,total_amount:double,congestio

In [25]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

'10485760b'

In [28]:
spark.conf.get("spark.sql.adaptive.enabled")

'true'

In [29]:
spark.conf.get("spark.sql.shuffle.partitions")

'200'

### Approximate Algorithms

In [10]:
yellow_tripdata.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2023-01-01 00:32:10|  2023-01-01 00:40:36|            1.0|         0.97|       1.0|                 N|         161|         141|           2|        9.3|  1.0|    0.5|       0.

In [12]:
yellow_tripdata.agg(F.approx_count_distinct("VendorID").alias("unique_vendors")).select(
    "unique_vendors"
).show()

26/09/24 12:25:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------+
|unique_vendors|
+--------------+
|             2|
+--------------+



In [13]:
yellow_tripdata.select("VendorID").distinct().count()

2

In [15]:
yellow_tripdata.select(
    F.percentile_approx("fare_amount", 0.95).alias("p95_amount")
).show()

+----------+
|p95_amount|
+----------+
|      65.3|
+----------+



In [16]:
spark.stop()